# Stage 2 Notebook 54 - Exp2YY Anchor + topk-fixed + VFL + full 70K + det rescue

**Combine all the wins on the full dataset.** This config aggregates everything we've learned:
- Anchor head with 192 priors (NB48 geometry champion: matched_iou=0.544).
- topk-fixed K=8 matching (Exp2WW's stability fix; replaces dynamic-k).
- VFL on continuous LineIoU regression target (Exp2QQ recipe).
- Full 70K BDD100K split (NB48 / NB51 scale).
- Aggressive det rescue: `lambda_det=3.0`, `lambda_lane=0.5`, `use_uncertainty=False` (NB51 used 2.0/0.7 and got minor improvement; this pushes harder).
- 6 epochs (full-data convergence is ~5 epochs based on NB48).

If Exp2WW (limit=3000) shows the topk-fixed matcher fixes the cls collapse on the anchor head, AND Exp2YY (this notebook, full data) inherits that win plus NB48-level geometry plus rescued det, this is the final stable model for Stage 3 deployment.

### Run mode
1. `DEBUG_MODE = True` smoke.
2. `DEBUG_MODE = False` for 6-epoch full-dataset run (~60-80 min).
3. Independent of all prior NBs. Run after Exp2WW (NB52) confirms the matcher fix works.
4. If OOM at full data, drop batch_size to 6.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint_smoke.log
OK exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.5352 det_loss=3.4451 grad_cos=0.0506 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5017282962799072, 'gate/lane_mean': 0.49809083342552185, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full6'
    EPOCHS = 6
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint_full6 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint_full6.tar --epochs 6 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint_full6.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp49_rmt_gca_anchor_topk_fixed_vfl_full_data_det_rescue_joint_full6_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --c

0

## What to watch in Exp2YY training

Reference NB48 (anchor + dynamic-k + VFL + 70K data): matched_iou=0.544, decoded_f1=0.05, gap=0.015, **val_det=3.16, val_map50=0**.
Reference NB51 (same + lambda_det=2.0): no improvement on det.

Pass criteria at epoch 6:
- **`val_det <= 2.5` and `val/det/map50 >= 0.005`** -- harder det rescue (lambda_det=3.0) actually trains det.
- **`val/matched_line_iou >= 0.50`** -- preserve geometry (acceptable small regression from NB48's 0.544).
- **`pos_score - neg_score >= 0.05`** -- topk-fixed matcher gives cls room to discriminate even at full data.
- **`val/lane/decoded_f1 >= 0.10`** -- 2x NB48; cls finally ranks correctly at full data scale.
- `train/grad_cosine_epoch_mean >= 0` across most epochs.

Failure signals:
- val_det still >= 3.0: lambda_det=3.0 not enough; the joint conflict at full data scale is structural. Pivot to PCGrad gradient surgery in Exp2ZZ.
- gap < 0.02 even with topk-fixed: matcher fix doesn't translate to full-data scale; the per-prior feature representation is the structural bottleneck. Pivot to query head (Exp2XX) at full data.
- matched_iou drops below 0.45: lambda_lane=0.5 too aggressive; raise to 0.7.